# 03 · SFT

Показать модели правильные ответы и подвинуть веса в их сторону. Для промпта $x$ и эталона $y$
из $T$ токенов:

$$
\mathcal{L}_{\text{SFT}}(\theta) = -\frac{1}{T}\sum_{t=1}^{T} \log \pi_\theta(y_t \mid x, y_{<t}).
$$

Сумма только по ответу: на токенах промпта в метках стоит $-100$, иначе модель училась бы писать
запросы студента. Учится не вся модель, а LoRA-адаптер: к каждому линейному слою прибавляется
$\frac{\alpha}{\sqrt r} B A$ ранга $r = 16$, около половины процента параметров. Множитель
$\alpha/\sqrt r$ — rsLoRA, при классическом $\alpha/r$ рост ранга душит обучение.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src import data, infer, metrics, report

from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

model, tokenizer = infer.load_model()
train = data.to_sft(data.load("train")).select_columns(["prompt", "completion"])
print(train)

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # attention and MLP projections of the language stack; the vision tower is excluded, there are no images
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True,
    task_type="CAUSAL_LM",
)

config = SFTConfig(
    output_dir=str(report.RUNS / "sft"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,               # a float is a share of total steps
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=2048,
    completion_only_loss=True,
    logging_steps=5,
    save_strategy="no",
    report_to=[],
    seed=42,
)
trainer = SFTTrainer(model=model, args=config, train_dataset=train, processing_class=tokenizer, peft_config=lora)
trainer.model.print_trainable_parameters()

## Проверка маски

Токены с $-100$ в `labels` градиента не дают. Граница должна лечь ровно после `<|im_start|>assistant`.

In [ ]:
example = trainer.train_dataset[0]
trained = [l != -100 for l in example["labels"]]
ids = example["input_ids"]
print(f"токенов {len(ids)}, под градиентом {sum(trained)}")
print("конец промпта:", repr(tokenizer.decode([i for i, t in zip(ids, trained) if not t])[-80:]))
print("начало ответа:", repr(tokenizer.decode([i for i, t in zip(ids, trained) if t])[:80]))

In [ ]:
history = trainer.train()
print(f"loss {history.training_loss:.3f}, {history.metrics['train_runtime'] / 60:.1f} мин")

tuned = trainer.model
tuned.save_pretrained(report.RUNS / "sft-adapter")
print(infer.free(trainer))
del trainer

## Замер

Тот же `report.evaluate`. `free` выше не косметика: состояние оптимизатора и служебные объекты тренера
иначе остаются в памяти и замер падает.

In [ ]:
results = report.evaluate(tuned, tokenizer, "sft", note="LoRA SFT, 3 epochs, lr 1e-4")
report.show()

In [ ]:
extended = list(data.load("test_extended"))
before = report.load_runs("extended")["base"]["answers"]
for i in (0, 13, 60):
    row = extended[i]
    print("═" * 78, row["id"], "·", data.request(row))
    print("ДО:\n" + before[row["id"]] + "\n\nПОСЛЕ:\n" + results["extended"]["answers"][i])

print()
print("регрессии к базе:", report.regressions("base", "sft") or "нет")